In [ ]:
# guess_country.py
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
# Init Chroma
chroma_client = chromadb.PersistentClient(".chroma_db")
collection = chroma_client.get_collection(name="wikipedia")

In [7]:
embedder = SentenceTransformer("intfloat/e5-base-v2")
flan = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use mps:0


In [22]:
statement = "This country is rich in oil resources."

In [30]:
# Get embeddings corresponding to statement
top_k = 5
results = collection.query(
    query_texts=[ statement ],
    n_results=top_k
)
print(results)

{'ids': [['Malaysia_26934', 'Algeria_762', 'Norway_33459', 'Venezuela_48685', 'Kyrgyzstan_24156']], 'embeddings': None, 'documents': [["one of the world's largest producers of palm oil.", '=== Oil and natural resources ===', '=== Resources ===\n\n\n==== Oil industry ====', '=== Petroleum and other resources ===', "Kumtor Gold Mine and other regions. The country's plentiful water resources and mountainous terrain enable it to produce and export large quantities of hydroelectric energy."]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'country': 'Malaysia'}, {'country': 'Algeria'}, {'country': 'Norway'}, {'country': 'Venezuela'}, {'country': 'Kyrgyzstan'}]], 'distances': [[0.26348215341567993, 0.27896198630332947, 0.28568583726882935, 0.28979748487472534, 0.29985153675079346]]}


In [ ]:
# print(results['metadatas'])

[[{'country': 'Malaysia'}, {'country': 'Algeria'}, {'country': 'Norway'}, {'country': 'Venezuela'}, {'country': 'Kyrgyzstan'}]]


In [34]:
countries = [ entry['country'] for entry in results['metadatas'][0] ]
print(countries)

['Malaysia', 'Algeria', 'Norway', 'Venezuela', 'Kyrgyzstan']


In [38]:
# Re-query DB filtering country
results = {}
for country in countries:
    country_results = collection.query(
        query_texts=[ statement ],
        n_results=top_k,
        where={'country': country}
    )
    country_texts = country_results['documents'][0]
    results[country] = country_texts
print(results)

{'Malaysia': ["one of the world's largest producers of palm oil.", "The country's economy has traditionally been driven by its natural resources but is expanding into commerce, tourism, and medical tourism. The country has a newly industrialised market economy, which is relatively open and state-oriented. The country is a founding member of the Organisation of Islamic Cooperation (OIC), the East Asia Summit (EAS), and the Association of Southeast Asian Nations (ASEAN), and a member of the Non-Aligned Movement (NAM), the Commonwealth, and the Asia-Pacific", 'and Herzegovina, Somalia, Kosovo, East Timor, and Lebanon.', "International trade, facilitated by the shipping route in adjacent Strait of Malacca, and manufacturing are the key sectors. Malaysia is an exporter of natural and agricultural resources, and petroleum is a major export. Malaysia has once been the largest producer of tin, rubber and palm oil in the world. Manufacturing has a large influence in the country's economy, altho

In [51]:
# Combine the contexts into one string per country
country_contexts = {}
for country, text in results.items():
    # print(country)
    context_str = ""
    for context in text:
        # print(context)
        context_str += f"{context}\n"
    # print(context_str)
    country_contexts[country] = context_str

In [52]:
# Summarize the contexts for each country in response to the statement
country_summaries = {}
for country, text in country_contexts.items():
    summary_prompt = (
        f"Given the following information about {country}:\n\n{text}\n"
        f"Summarize how {country} relates to the statement: \"{statement}\""
    )
    summary = flan(summary_prompt, do_sample=False)[0]['generated_text']
    country_summaries[country] = summary
    print(f"{country} summary:\n{summary}\n")

Malaysia summary:
The country's economy has traditionally been driven by its natural resources but is expanding into commerce, tourism, and medical tourism.

Algeria summary:
The relevant information to answer the above question is: Algeria is a semi-presidential republic composed of 58 provinces (wilayas) and 1,541 communes. It is a regional power in North Africa and a middle power in global affairs. As of 2025, the country has the highest Human Development Index in continental Africa, due mostly to its large petroleum and natural gas reserves, which are the sixteenth and ninth largest in the world, respectively. Sonatrach, the national oil company, is the largest company in Africa oil company, is the largest company in Africa and a major supplier of natural gas to Europe.

Norway summary:
The country has the fourth- and eighth-highest per-capita income in the world on the World Bank's and IMF's list, respectively. It has the world's largest sovereign wealth fund, with a value of US$1

In [53]:
def get_per_country_summary_for_statement(statement: str, top_k: int = 5) -> str:
    # Get embeddings corresponding to statement
    results = collection.query(
        query_texts=[ statement ],
        n_results=top_k
    )

    # Get countries mentioned in embeddings
    countries = [ entry['country'] for entry in results['metadatas'][0] ]

    # Re-query DB filtering country
    results = {}
    for country in countries:
        country_results = collection.query(
            query_texts=[ statement ],
            n_results=top_k,
            where={'country': country}
        )
        country_texts = country_results['documents'][0]
        results[country] = country_texts
    
    # Combine the contexts into one string per country
    country_contexts = {}
    for country, text in results.items():
        context_str = ""
        for context in text:
            context_str += f"{context}\n"
        country_contexts[country] = context_str

    # Summarize the contexts for each country in response to the statement
    country_summaries = {}
    final_result = ""
    for country, text in country_contexts.items():
        summary_prompt = (
            f"Given the following information about {country}:\n\n{text}\n"
            f"Summarize how {country} relates to the statement: \"{statement}\""
        )
        summary = flan(summary_prompt, do_sample=False)[0]['generated_text']
        country_summaries[country] = summary
        final_result += f"{country} summary:\n{summary}\n"
    
    return final_result

In [55]:
print(get_per_country_summary_for_statement("This country is hot all year round."))

Tanzania summary:
The hottest period extends between November and February (25–31 °C or 77.0–87.8 °F) while the coldest period occurs between May and August (15–20 °C or 59–68 °F). The hottest period extends between November and February (25–31 °C or 77.0–87.8 °F) while the coldest period occurs between May and August (15–20 °C or 59–68 °F). The hottest period extends between November and February (25–31 °C or 77.0–87.8 °F) while the coldest period occurs between May and August (15–20 °C or 59–68 °F). The hottest period extends between November and February (25–31 °C or 77.0–87.8 °F) while the coldest period occurs between May and August (15–20 °C or 59–68 °F). The hottest period extends between November and February (25–31 °C or 77.0–87.8 °F) while the coldest period occurs between May and August (15–20 °C
Zambia summary:
The relevant sentence in the passage is: However, average monthly temperatures remain above 20 °C (68 °F) over most of the country for eight or more months of the ye

In [56]:
print(get_per_country_summary_for_statement("This country does not have seasons."))

Tanzania summary:
Tanzania is located entirely south of the equator.
Reunion summary:
The country Reunion is in is not a country that has seasons.
Rwanda summary:
The relevant sentence in the passage is: There are two rainy seasons in the year; the first runs from February to June and the second from September to December. The relevant sentence in the passage is: There are two rainy seasons in the year; the first runs from February to June and the second from September to December. The relevant sentence in the passage is: There are two dry seasons in the year; the first runs from February to June and the second from September to December. The relevant sentence in the passage is: There are two rainy seasons in the year; the first runs from February to June and the second from September to December. The relevant sentence in the passage is: There are two rainy seasons in the year; the first runs from February to June and the second from September to December. The relevant sentence in the 

In [57]:
print(get_per_country_summary_for_statement("This country became independent after 1950."))

Singapore summary:
The country became an independent sovereign country in 1965.
Lebanon summary:
The country became independent after 1950.
Canada summary:
The delay underscored Canada's independence.
Solomon Islands summary:
The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon Islands. The following is the history of Solomon